<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 7 — Sampling the Posterior</div>
</div>

#### Bayesian inference with your own Metropolis–Hastings sampler: likelihoods and priors, checking the chain, and credible intervals, on a century-old sunflower.
***

## Where this fits

Lecture 8 turned a fit into a **distribution**: Bayes' theorem gives the posterior, what we know about the parameters after seeing the data, and Metropolis–Hastings draws samples from it when a grid would be too costly. Today you build that sampler yourself, check that it works, and read the answer off its samples.

The data: the height of a sunflower measured every week for 12 weeks, a classic dataset from 1919. Plants grow slowly, then fast, then level off, a curve with three parameters that you will infer:

0. look at the data, and a quick least-squares fit to compare with later;
1. the log-posterior: likelihood and priors in one function;
2. the Metropolis–Hastings sampler (you write the accept step);
3. is it working? trace plots, burn-in, step size, and how many *independent* samples you really have;
4. the answer: credible intervals, correlations, and curves drawn from the posterior; then, what if you don't know the uncertainty?

Samplers like this one come back later in the module. **Nothing here is assessed.**

## How to use this notebook today

This is the week Exercise 2 is due, so the lab is deliberately short. **Core (about 60 minutes):** Sections 0–4. The core cells contain most of the code: fill in each `____`, run it, and compare with the **Check**. Everything marked *Optional* can wait.

**New to some of the Python?** Start with `week7_python_toolkit.ipynb` (15 minutes): random numbers, why we work with log-probabilities, percentiles, and the autocorrelation of a sequence. If the basics themselves are shaky, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises them one at a time; each section below names the ones it uses.

Each task has a **Check** with the number you should get, and some have a collapsed hint.

In [ ]:
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week7/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

NAMES = ["K (cm)", "r (per day)", "t0 (day)"]

## <span style="color:#B5121B">0. The data and the model (10 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 6 (functions); Section 8 (plots that say something). Each section takes 10–15 minutes.

The growth follows an S-shaped (**logistic**) curve,
$$h(t) = \frac{K}{1 + e^{-r\,(t - t_0)}},$$
where $K$ is the final height, $r$ the growth rate, and $t_0$ the day of fastest growth (when the plant is half its final height).

**(a)** Load the data and plot height against day. Leave out day 0, the planting day: no logistic curve can pass through a height of exactly 0, so that point is not a measurement of this model.

**(b)** No uncertainties are given. **For now, assume each height is uncertain by $\sigma = 4$ cm.** Fit the logistic curve by weighted least squares (`curve_fit`, Lecture 3), and report $K$, $r$ and $t_0$ with their uncertainties and $\chi^2/\nu$. Keep these numbers: you'll compare the sampler with them.

In [ ]:
flower = pd.read_csv(data_path("sunflower_growth.csv"))
grow = flower[flower["day"] > 0]                       # leave out the planting day
t, h = grow["day"].to_numpy(dtype=float), grow["height_cm"].to_numpy()
SIGMA = 4.0                                            # assumed uncertainty of each height (cm)

def logistic(t, K, r, t0):
    """Height (cm) on day t: final height K, growth rate r (per day), midpoint t0 (day)."""
    return K / (1 + np.exp(-r * (t - t0)))

In [ ]:
p_ls, cov_ls = curve_fit(logistic, t, h, p0=[250, 0.1, 35], sigma=np.full(len(t), ____), absolute_sigma=True)
err_ls = np.sqrt(np.diag(cov_ls))
chi2_nu = np.sum(((h - logistic(t, *p_ls)) / SIGMA)**2) / (len(t) - 3)
for name, p, e in zip(NAMES, p_ls, err_ls):
    print(f"{name:12s} {p:.4g} ± {e:.2g}")
print(f"chi2/nu = {chi2_nu:.2f}")

tt = np.linspace(0, 90, 300)
plt.errorbar(t, h, SIGMA, fmt="o", color="k", ms=4, label="data (σ = 4 cm assumed)")
plt.plot(tt, logistic(tt, *p_ls), color="#B5121B", label="least-squares fit")
plt.xlabel("Day"); plt.ylabel("Height (cm)"); plt.legend(); plt.show()

**Check:** $K = 261.0 \pm 2.8$ cm, $r = 0.0877 \pm 0.0032$ per day, $t_0 = 34.3 \pm 0.5$ days, and $\chi^2/\nu = 0.88$: 4 cm is a reasonable guess at the scatter.

## <span style="color:#B5121B">1. The log-posterior (10 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 6 (functions). Each section takes 10–15 minutes.

Bayes' theorem (Lecture 8): posterior $\propto$ likelihood $\times$ prior. We work with **logarithms**: products of many small probabilities underflow to zero, sums of their logs don't, and the sampler only ever needs *differences* of log-posteriors.

- **Likelihood.** Each height is the model plus Gaussian noise of standard deviation $\sigma$, independently. Then, up to a constant, $\ln L = -\tfrac{1}{2}\sum_i \left(\frac{h_i - h(t_i)}{\sigma}\right)^2 = -\tfrac{1}{2}\chi^2$.
- **Prior.** Flat (uniform) within physically sensible limits, zero outside: $100 < K < 500$ cm, $0 < r < 1$ per day, $0 < t_0 < 84$ days. Outside the limits, $\ln(\text{prior}) = -\infty$.

**(a)** Complete `log_post(theta)`, where `theta = [K, r, t0]`. **(b)** Evaluate it at the least-squares values and at `[200, 0.1, 40]`. Which is more probable, and by what factor?

In [ ]:
def log_post(theta):
    """Log-posterior (up to a constant) for theta = [K, r, t0], with sigma fixed at SIGMA."""
    K, r, t0 = theta
    if not (100 < K < 500 and 0 < r < 1 and 0 < t0 < 84):
        return -np.inf                                  # outside the prior: impossible
    resid = (h - logistic(t, K, r, t0)) / SIGMA
    return ____                                         # -chi^2 / 2

lp_best, lp_other = log_post(p_ls), log_post([200, 0.1, 40])
print(f"log-posterior at the fit {lp_best:.2f}; at [200, 0.1, 40] {lp_other:.1f}")
print(f"the fit is e^{lp_best - lp_other:.0f} times more probable")

**Check:** about −4.0 at the least-squares values (that is, $-\chi^2/2$ with $\chi^2 = 7.9$), and about −1064 at `[200, 0.1, 40]`: a factor of roughly $e^{1060}$, far beyond what a computer can represent as a plain probability. That is why we use logs.

## <span style="color:#B5121B">2. The Metropolis–Hastings sampler (10 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 5 (loops that fill an array); Section 7 (functions as inputs). Each section takes 10–15 minutes.

> **Primer: Metropolis–Hastings** (Lecture 8). From the current point $\theta_n$, propose $\theta' = \theta_n + $ a random Gaussian step. Accept it with probability $\min\left(1, \frac{p(\theta' \mid d)}{p(\theta_n \mid d)}\right)$; if rejected, stay at $\theta_n$ **and count it again**. In logs: accept if $\ln u < \ln p(\theta') - \ln p(\theta_n)$, where $u$ is a uniform random number between 0 and 1. Uphill moves are always accepted, downhill moves sometimes, and in the long run each point is visited in proportion to its posterior probability.

The loop is written for you. **Write the accept step** (one line), then run 20 000 steps from a deliberately poor start, `[150, 0.3, 60]`, with step widths of 3 cm, 0.004 per day and 0.6 days.

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

The condition is `np.log(rng.uniform()) < lp_prop - lp`. Taking the log of $u$ instead of exponentiating the difference avoids overflow when the difference is large.

</details>

In [ ]:
def metropolis(log_post, theta0, steps, n, seed=1):
    """Metropolis-Hastings: n samples from exp(log_post), Gaussian proposals with widths `steps`.
    Returns the chain (n x number of parameters) and the fraction of proposals accepted."""
    rng = np.random.default_rng(seed)
    theta = np.array(theta0, dtype=float)
    lp = log_post(theta)
    chain = np.empty((n, len(theta)))
    accepted = 0
    for i in range(n):
        prop = theta + steps * rng.standard_normal(len(theta))      # propose
        lp_prop = log_post(prop)
        if ____:                                                    # accept?
            theta, lp = prop, lp_prop
            accepted += 1
        chain[i] = theta                                            # a rejected step counts again
    return chain, accepted / n

STEPS = np.array([3.0, 0.004, 0.6])
chain, acc = metropolis(log_post, [150, 0.3, 60], STEPS, 20000)
print(f"accepted {acc:.0%} of proposals; last sample {np.round(chain[-1], 3)}")

**Check:** about 25% of proposals accepted, close to Lecture 8's rule of thumb (slide 22): about 23% for several parameters, 44% for one. The run takes a second or two.

## <span style="color:#B5121B">3. Is it working? (15 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 3 (indexing and slicing); Section 8 (plots that say something). Each section takes 10–15 minutes.

A sampler always returns numbers. Before believing them, check three things.

**(a) Trace and burn-in.** Plot each parameter against step number. The chain is a dynamical process we built: from a poor start there is a **transient** (the *burn-in*) before it settles. How many steps does it take? Discard that stretch; keep everything after step 2000. After the transient, any stretch of the trace should have the same statistics (it is *stationary*); a slow drift means the transient is not over (Lecture 8, slide 22).

**(b) Step width.** Rerun with all steps 10 times smaller, and 10 times larger. What fraction is accepted in each case, and what do the traces look like?

**(c) How many independent samples?** Successive samples are not independent: each starts from the one before. The **autocorrelation** of the chain at lag $\ell$ measures how alike samples $\ell$ steps apart are; the **effective sample size** is
$$N_\mathrm{eff} = \frac{N}{1 + 2\sum_{\ell\ge1}\rho_\ell},$$
summing until $\rho_\ell$ falls below about 0.05 (Lectures 6 and 9 write the lag as $\ell$). (This is the same idea as Lab 5, Section 0 and Exercise 2, Task 1.2: correlated values carry less information than their number suggests.) Compute $N_\mathrm{eff}$ for $r$ for the three step widths.

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

A function for the autocorrelation, using the FFT (fast for long chains):
```python
def autocorr(x):
    x = x - x.mean()
    f = np.fft.rfft(x, 2 * len(x))
    ac = np.fft.irfft(f * np.conj(f))[:len(x)]
    return ac / ac[0]
```
Then `rho = autocorr(chain[2000:, 1])`, and add up `rho[1:]` until the first value below 0.05.

</details>

In [ ]:
def autocorr(x):
    """Autocorrelation of a sequence at lags 0, 1, 2, ... (via the FFT)."""
    x = x - x.mean()
    f = np.fft.rfft(x, 2 * len(x))
    ac = np.fft.irfft(f * np.conj(f))[:len(x)]
    return ac / ac[0]

def n_eff(x, cut=0.05):
    """Effective number of independent samples in x."""
    rho = autocorr(x)
    k = np.argmax(rho < cut)              # first lag where the autocorrelation has died away
    return len(x) / (1 + 2 * rho[1:k].sum())

BURN = 2000
fig, ax = plt.subplots(3, 3, figsize=(12, 6), sharex=True)
for j, scale in enumerate([0.1, 1, 10]):
    ch, a = metropolis(log_post, [150, 0.3, 60], ____ * STEPS, 20000)
    for i in range(3):
        ax[i, j].plot(ch[:, i], lw=0.5)
        ax[i, 0].set_ylabel(NAMES[i])
    ax[0, j].set_title(f"steps x {scale}: {a:.0%} accepted, N_eff(r) = {n_eff(ch[BURN:, 1]):.0f}")
for a_ in ax[-1]: a_.set_xlabel("step")
plt.tight_layout(); plt.show()

**Check:** the chain arrives within about 300 steps, so discarding 2000 is safe. Tiny steps: about 85% accepted but the chain creeps, and $N_\mathrm{eff} \approx 13$ of 18 000. Sensible steps: about 25% accepted, $N_\mathrm{eff} \approx 900$. Huge steps: almost nothing accepted (under 1%), and $N_\mathrm{eff} \approx 20$. Only the middle one has explored the posterior properly.

## <span style="color:#B5121B">4. The answer (15 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 3 (indexing and slicing); Section 8 (plots that say something). Each section takes 10–15 minutes.

**(a)** From the chain after burn-in, give the **median** and the **68% credible interval** (the 16th to 84th percentiles) of $K$, $r$ and $t_0$. Compare them with the least-squares values from Section 0.

**(b)** Plot $t_0$ against $K$, one point per sample. What does the tilt tell you? (Print the correlation matrix of the samples.)

**(c)** Draw 100 samples at random from the chain and plot their growth curves over the data: the spread of curves is the uncertainty of the prediction.

**(d) What if you don't know $\sigma$?** Make it a fourth parameter, with a flat prior $0 < \sigma < 50$ cm. The log-likelihood now needs one extra term, $-N\ln\sigma$, which you could drop while $\sigma$ was fixed: without it, the sampler would make $\sigma$ huge, because a large $\sigma$ makes every residual look small. Run the sampler (step width 0.6 cm for $\sigma$, 40 000 steps) and compare the inferred $\sigma$ with the 4 cm you assumed.

In [ ]:
post = chain[BURN:]
lo, med, hi = np.percentile(post, [16, 50, 84], axis=0)
for i, name in enumerate(NAMES):
    print(f"{name:12s} {med[i]:.4g}  (68%: {lo[i]:.4g} to {hi[i]:.4g});  least squares {p_ls[i]:.4g} ± {err_ls[i]:.2g}")
print("correlation of K, r, t0:\n", np.corrcoef(post.T).round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(post[::10, 0], post[::10, 2], ".", ms=2, alpha=0.4)
ax[0].set(xlabel=NAMES[0], ylabel=NAMES[2])
rng = np.random.default_rng(3)
for k in rng.choice(len(post), 100, replace=False):
    ax[1].plot(tt, logistic(tt, *post[k]), color="#B5121B", alpha=0.05)
ax[1].errorbar(t, h, SIGMA, fmt="o", color="k", ms=4)
ax[1].set(xlabel="Day", ylabel="Height (cm)")
plt.tight_layout(); plt.show()

def log_post4(theta):
    """Log-posterior for theta = [K, r, t0, sigma]: sigma is now a parameter."""
    K, r, t0, sigma = theta
    if not (100 < K < 500 and 0 < r < 1 and 0 < t0 < 84 and 0 < sigma < 50):
        return -np.inf
    resid = (h - logistic(t, K, r, t0)) / sigma
    return -0.5 * np.sum(resid**2) - len(t) * ____         # the extra term

chain4, acc4 = metropolis(log_post4, [150, 0.3, 60, 10], np.array([3.0, 0.004, 0.6, 0.6]), 40000, seed=2)
lo4, med4, hi4 = np.percentile(chain4[4000:], [16, 50, 84], axis=0)
print(f"sigma: {med4[3]:.2f} cm (68%: {lo4[3]:.2f} to {hi4[3]:.2f}); {acc4:.0%} accepted")

**Check:** $K = 261.0$ cm (68%: 258.2 to 263.6), $r = 0.088$ (0.085 to 0.091), $t_0 = 34.3$ (33.8 to 34.7): the same as least squares, within the sampling noise, because with Gaussian errors and flat priors the posterior is the least-squares answer. $K$ and $t_0$ are correlated (+0.71): a taller final plant needs a later midpoint to fit the same points. With $\sigma$ free: $\sigma = 4.2$ cm (68%: 3.3 to 5.7), so 4 cm was a fair guess; the other parameters barely change.

## <span style="color:#FD9029">5. Optional: emcee, checked with ArviZ</span>

This is the practical Lecture 9 promised ("What you will type"). `emcee` runs many *walkers* at once and adapts to correlated posteriors; `ArviZ` reports the checks ($\hat R$ as `r_hat`, $N_\mathrm{eff}$ as `ess_bulk`). The toolkit's Section 6 shows both on a straight line. Sample `log_post` with 32 walkers for 3000 steps, discard the first 500, and compare the medians and 68% intervals with your own sampler. Then print ArviZ's diagnostics. Is $\hat R$ 1.01 or less? If not, run longer (10 000 steps) and check again. How many effective samples do you have, compared with your own sampler in Section 3?

In [ ]:
# Optional

## <span style="color:#FD9029">6. Optional: several chains</span>

One chain can look settled and still be stuck. Run four chains from very different starts, and compare the spread *within* each chain with the spread *between* their means: the Gelman–Rubin statistic $\hat R = \sqrt{\left[\frac{N-1}{N}W + \frac{B}{N}\right] / W}$, where $W$ is the mean within-chain variance and $B/N$ the variance of the chain means. Values close to 1 (below about 1.01) mean the chains agree.

In [ ]:
# Optional

## <span style="color:#FD9029">7. Optional: does the prior matter?</span>

Suppose the seed packet says this variety grows to about 3 m: a prior $K \sim \mathcal{N}(300, 10)$ cm. Add $-\tfrac12\left(\frac{K-300}{10}\right)^2$ to the log-posterior and rerun. How far does $K$ move, and why so little (or so much)? Then repeat with only the first six weeks of data (days 7–42). Lecture 8, slide 16: many data, the data decide; few data, the prior pulls the posterior.

In [ ]:
# Optional

## Summary

- **Log-posterior** = log-likelihood + log-prior. With Gaussian errors the log-likelihood is $-\chi^2/2$; work in logs to avoid underflow.
- **Metropolis–Hastings:** propose, accept with probability $\min(1, p'/p)$, and count a rejected step again. Uphill always, downhill sometimes.
- **Check before you believe:** look at the traces, discard the burn-in, aim for an acceptance rate near 23% for several parameters (44% for one), and count the *effective* samples: correlated samples carry less information than their number.
- **Report** medians and credible intervals, and the correlations between parameters; draw predictions from the posterior.
- **Unknown noise** can be a parameter too, but its likelihood needs the $-N\ln\sigma$ term.
- With Gaussian errors and flat priors, the posterior agrees with least squares. The sampler earns its keep when the posterior isn't Gaussian, the priors matter, or there are too many parameters for a grid.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>